## Evaluation and minimization of the loss function

In [1]:
import numpy as np
from scipy.optimize import minimize

In [2]:
from ph_refine import load_ph_data, ph_loss, ph_tilde_loss_and_grad, Lambdas

### 1st step: load data from experiments and MD simulations

In [3]:
ph_vals = {'A3mer' : [3.00, 3.50, 4.00], 'A5mer' : [3.50, 4.00, 4.50]}

mol_name = 'A3mer'

data = load_ph_data(path='Simulation-data', mol_name=mol_name, obs_names=['chi', 'eRMSD'],
                 ph_vals=ph_vals[mol_name], ref_ph=4.00, g_exp=None, sigma_exp=None)

In [4]:
data

          g_exp: Array([-0.17252229, -0.22463956, -0.26805093,  1.07583431,  1.02142011,
                         1.03818499], dtype=float64)
             gs: {0: Array([[-2.291882,  0.369919],
                        [-1.462641,  0.68909 ],
                        [-2.415568,  0.666532],
                        ...,
                        [-1.283432,  1.146141],
                        [-0.592147,  1.194844],
                        [-0.692991,  1.196314]], dtype=float64), 1: Array([[-2.549456,  0.14379 ],
                        [-1.78495 ,  0.457525],
                        [ 2.767067,  1.019181],
                        ...,
                        [-0.931121,  1.180245],
                        [-0.388632,  1.166304],
                        [-0.803565,  1.224548]], dtype=float64)}
  legend_matrix: Array([[0, 1, 2],
                        [3, 4, 5]], dtype=int32)
 legend_weights: {0: array([    0, 13633, 43603, 93598]), 1: array([     0,  61367, 106397, 131402])}
 log_fugacitie

### 2nd step: evaluate the loss function for the original `data`

In [16]:
pi_vec = data.pops[data.ref_ph]
lambdas_vec = np.zeros(len(data.g_exp))

out = ph_loss(lambdas=lambdas_vec, pis=pi_vec, data=data)

In [17]:
out

         avs: Array([[-0.57381776,  1.06441976],
                     [ 0.53258331,  1.22029273]], dtype=float64)
      avs_ph: Array([[ 0.36991613,  0.1422681 , -0.16759854],
                     [ 1.19737571,  1.165304  ,  1.12164909]], dtype=float64)
 check_gamma: Array(13965.53592117, dtype=float64)
        chi2: Array(27931.07184235, dtype=float64)
       dkl_p: Array([-0., -0.], dtype=float64)
      dkl_pi: Array(0., dtype=float64)
       gamma: Array(0., dtype=float64)
      log_ps: [Array([-12.96988773, -14.43802518, -14.25847303, ..., -10.18431453,
                     -11.91815063, -11.22710675], dtype=float64), Array([-17.59996934, -13.70569694, -13.28922068, ..., -10.62249164,
                     -14.05824754, -11.25854975], dtype=float64)]
        loss: Array(13965.53592117, dtype=float64)
    rel_diff: Array([[78.08238488, 52.62060835, 14.53094746],
                     [82.92584501, 94.6325298 , 54.9738546 ]], dtype=float64)

In [18]:
data.g_exp

Array([-0.17252229, -0.22463956, -0.26805093,  1.07583431,  1.02142011,
        1.03818499], dtype=float64)

### 3rd step: minimize the loss function through `ph_tilde_loss`
(dimensionality reduction with `lambdas`)

In [7]:
log_pi_vec = np.log(pi_vec)


In [8]:
# is_fixed = False, so inner minimization of Gamma (required to compute the gradient)
lambdas = Lambdas(np.zeros(len(data.g_exp)), False)

ph_tilde_loss_and_grad(log_pi_vec, data, lambdas)


loss, grad:  216.37605158291205 [-47.88484792  47.88484792]


(Array(216.37605158, dtype=float64),
 Array([-47.88484792,  47.88484792], dtype=float64))

In [9]:
args = (data, lambdas)

mini = minimize(ph_tilde_loss_and_grad, log_pi_vec, args=args, method='BFGS', jac=True)


loss, grad:  216.37605158291205 [-47.88484792  47.88484792]
loss, grad:  145.4357826672163 [-41.85227717  41.85227717]
loss, grad:  134.08477354468442 [ 112.69000544 -112.69000544]
loss, grad:  115.92243984956265 [-22.3248032  22.3248032]
loss, grad:  100.80768868118658 [-7.1865594  7.1865594]
loss, grad:  99.28284250247877 [ 2.26586942 -2.26586942]
loss, grad:  99.19469885312417 [-0.70321786  0.70321786]
loss, grad:  99.18381056200413 [-0.05398744  0.05398744]
loss, grad:  99.18374748555367 [ 0.00109152 -0.00109152]
loss, grad:  99.18374744619904 [ 0.00058007 -0.00058007]
loss, grad:  99.18374743881672 [-0.00015549  0.00015549]
loss, grad:  99.18374743776104 [-3.21977804e-05  3.21977804e-05]
loss, grad:  99.18374743771203 [-7.70100982e-07  7.70100982e-07]


In [10]:
mini

  message: Optimization terminated successfully.
  success: True
   status: 0
      fun: 99.18374743771203
        x: [ 1.457e+00 -2.917e+00]
      nit: 11
      jac: [-7.701e-07  7.701e-07]
 hess_inv: [[ 5.229e-01  4.771e-01]
            [ 4.771e-01  5.229e-01]]
     nfev: 13
     njev: 13

In [11]:
x = mini.x

pi_new = np.exp(x)
pi_new /= np.sum(pi_new)

pi_new

array([0.98755126, 0.01244874])

In [12]:
vars(lambdas)

{'value': array([   63.44504637,  -238.21934852,   174.73622681, -2014.82257509,
         7181.30896255, -5180.17489963]),
 'is_fixed': False}

### 4. evaluate `ph_loss` on the optimal solution
- do the values of the loss function agree?
- is this value lower than the starting one?

In [13]:
out = ph_loss(lambdas.value, pi_new, data)

out

         avs: Array([[-0.27098807,  1.02057517],
                     [ 0.63592551,  1.47552421]], dtype=float64)
      avs_ph: Array([[-0.16946343, -0.23622199, -0.25969813],
                     [ 1.07150454,  1.03801541,  1.02623871]], dtype=float64)
 check_gamma: Array(1.56530074e-05, dtype=float64)
        chi2: Array(194.18415729, dtype=float64)
       dkl_p: Array([0.05165116, 1.64270076], dtype=float64)
      dkl_pi: Array(0.39733254, dtype=float64)
       gamma: Array(-98.7864149, dtype=float64)
      log_ps: [Array([-12.60232643, -14.25519936, -14.22685102, ..., -10.45121669,
                     -12.10951399, -11.43854452], dtype=float64), Array([-35.12925546, -26.72042384, -18.68536016, ..., -13.10538219,
                     -16.82567524, -13.106786  ], dtype=float64)]
        loss: Array(99.18376309, dtype=float64)
    rel_diff: Array([[ 0.44031375, -1.66111095,  1.20827405],
                     [-2.95413748, 10.91474389, -7.86844985]], dtype=float64)

In [14]:
print(mini.fun, out.loss)

99.18374743771203 99.18376309072038


yes, they agree!

In [15]:
out

         avs: Array([[-0.27098807,  1.02057517],
                     [ 0.63592551,  1.47552421]], dtype=float64)
      avs_ph: Array([[-0.16946343, -0.23622199, -0.25969813],
                     [ 1.07150454,  1.03801541,  1.02623871]], dtype=float64)
 check_gamma: Array(1.56530074e-05, dtype=float64)
        chi2: Array(194.18415729, dtype=float64)
       dkl_p: Array([0.05165116, 1.64270076], dtype=float64)
      dkl_pi: Array(0.39733254, dtype=float64)
       gamma: Array(-98.7864149, dtype=float64)
      log_ps: [Array([-12.60232643, -14.25519936, -14.22685102, ..., -10.45121669,
                     -12.10951399, -11.43854452], dtype=float64), Array([-35.12925546, -26.72042384, -18.68536016, ..., -13.10538219,
                     -16.82567524, -13.106786  ], dtype=float64)]
        loss: Array(99.18376309, dtype=float64)
    rel_diff: Array([[ 0.44031375, -1.66111095,  1.20827405],
                     [-2.95413748, 10.91474389, -7.86844985]], dtype=float64)

#### 5. ok, it works, but...
why there is such a big discrepancy in the original data between averages over MD simulations and experimental values, where the latter are synthetic, given by average values over MD??

In [19]:
data.g_exp

Array([-0.17252229, -0.22463956, -0.26805093,  1.07583431,  1.02142011,
        1.03818499], dtype=float64)

In [20]:
pi_vec = data.pops[data.ref_ph]
lambdas_vec = np.zeros(len(data.g_exp))

out = ph_loss(lambdas=lambdas_vec, pis=pi_vec, data=data)

out

         avs: Array([[-0.57381776,  1.06441976],
                     [ 0.53258331,  1.22029273]], dtype=float64)
      avs_ph: Array([[ 0.36991613,  0.1422681 , -0.16759854],
                     [ 1.19737571,  1.165304  ,  1.12164909]], dtype=float64)
 check_gamma: Array(13965.53592117, dtype=float64)
        chi2: Array(27931.07184235, dtype=float64)
       dkl_p: Array([-0., -0.], dtype=float64)
      dkl_pi: Array(0., dtype=float64)
       gamma: Array(0., dtype=float64)
      log_ps: [Array([-12.96988773, -14.43802518, -14.25847303, ..., -10.18431453,
                     -11.91815063, -11.22710675], dtype=float64), Array([-17.59996934, -13.70569694, -13.28922068, ..., -10.62249164,
                     -14.05824754, -11.25854975], dtype=float64)]
        loss: Array(13965.53592117, dtype=float64)
    rel_diff: Array([[78.08238488, 52.62060835, 14.53094746],
                     [82.92584501, 94.6325298 , 54.9738546 ]], dtype=float64)

In [26]:
print('experimental values (synthetic): ', data.g_exp)
print('\naverage values at fixed pH: \n', out.avs_ph)

experimental values (synthetic):  [-0.17252229 -0.22463956 -0.26805093  1.07583431  1.02142011  1.03818499]

average values at fixed pH: 
 [[ 0.36991613  0.1422681  -0.16759854]
 [ 1.19737571  1.165304    1.12164909]]


I guess this is due to the fact that, in the second case, the weights of different protonation states are computed through the grand-canonical statistics and not read from all the MD simulations